# 05b — SHAP + LLM Explainer
**Domain:** Healthcare / Heart Disease UCI &nbsp;|&nbsp; **Time:** ~1.5 h  
**Key Concepts:** SHAP TreeExplainer, feature importance, natural-language explanations


In [ ]:
import sys, os
sys.path.insert(0, '.')
from llm_checker import check, hint, evaluate, progress, show_exercise
print("✅ ready")


In [ ]:
import pandas as pd, numpy as np

HEART_PATH = os.path.join("data", "heart_disease", "heart.csv")
if os.path.exists(HEART_PATH):
    heart_df = pd.read_csv(HEART_PATH)
    print(f"✅ Loaded {len(heart_df)} rows — columns: {heart_df.columns.tolist()}")
else:
    print("⚠️  Heart disease data not found — using synthetic fallback")
    rng = np.random.default_rng(42)
    n = 303
    heart_df = pd.DataFrame({
        "age":      rng.integers(30, 80, n),
        "sex":      rng.integers(0, 2, n),
        "cp":       rng.integers(0, 4, n),
        "trestbps": rng.integers(90, 200, n),
        "chol":     rng.integers(150, 400, n),
        "fbs":      rng.integers(0, 2, n),
        "thalach":  rng.integers(80, 200, n),
        "exang":    rng.integers(0, 2, n),
        "oldpeak":  rng.uniform(0, 6, n).round(1),
        "target":   rng.integers(0, 2, n),
    })


---
## 🔵 Example — Ex 05b-1: SHAP Values + Natural-Language Summary

In [ ]:
from xgboost import XGBClassifier
import shap
from openai import OpenAI

client = OpenAI(base_url="http://localhost:1234/v1", api_key="lm-studio")

feature_cols = [c for c in heart_df.columns if c != "target"]
X = heart_df[feature_cols].values
y = heart_df["target"].values

model = XGBClassifier(n_estimators=100, max_depth=3, random_state=42, eval_metric="logloss")
model.fit(X, y)

explainer   = shap.TreeExplainer(model)
shap_values = explainer(X)
print(f"SHAP matrix shape: {shap_values.values.shape}")

def shap_summary_for(patient_idx: int) -> str:
    sv   = shap_values[patient_idx].values
    top5 = sorted(zip(feature_cols, sv), key=lambda x: abs(x[1]), reverse=True)[:5]
    risk = model.predict_proba(X[patient_idx:patient_idx+1])[0][1]
    lines = "\n".join(f"  {f}: SHAP={v:+.3f}" for f, v in top5)
    prompt = (
        f"Patient predicted cardiac risk = {risk:.0%}.\n"
        f"Top SHAP features (positive = increases risk):\n{lines}\n"
        "Explain in 2 plain-English sentences suitable for a clinician."
    )
    resp = client.chat.completions.create(
        model="local-model",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=100,
    )
    return resp.choices[0].message.content.strip()

for idx in [0, 5, 10]:
    print(f"--- Patient {idx} ---")
    print(shap_summary_for(idx))
    print()


---
## 🟡 Exercise — Ex 05b-2: explain_prediction() Wrapper

In [ ]:
show_exercise(
    "05b-2", "explain_prediction() for 20 patients",
    """Write explain_prediction(patient_row: pd.Series) -> dict with keys:
  predicted_risk  (float in [0, 1])
  top_features    (list[str] — names of top 3 SHAP features)
  explanation     (str — plain-English sentence for a doctor)
Run on 20 patients and verify >= 80% of explanations mention at least one feature name.""",
    hints=[
        "patient_row.values.reshape(1, -1) to get model input",
        "Use shap_summary_for() from the example as a starting point",
        "Check: any(f in explanation for f in feature_cols)",
    ],
    checks=[
        "explain_prediction() returns dict with 3 keys",
        "predicted_risk is float in [0, 1]",
        "top_features is a list of strings",
        ">= 80% of 20 explanations mention at least one feature name",
    ],
)


In [ ]:
def explain_prediction(patient_row: "pd.Series") -> dict:
    """SHAP + LLM explanation for one patient."""
    # TODO: get predicted_risk from model.predict_proba
    # TODO: compute SHAP values for this patient
    # TODO: identify top 3 features by |SHAP|
    # TODO: generate explanation string with LLM
    # Return: {"predicted_risk": float, "top_features": list[str], "explanation": str}
    pass


explanations = []
for i in range(20):
    row = heart_df.iloc[i]
    try:
        explanations.append(explain_prediction(row))
    except Exception as e:
        print(f"Patient {i} failed: {e}")


In [ ]:
grounded = sum(
    1 for e in explanations
    if e and any(f in e.get("explanation", "") for f in feature_cols)
)
grounded_pct = grounded / max(len(explanations), 1)
print(f"Grounded: {grounded}/{len(explanations)} ({grounded_pct:.0%})")

check([
    (len(explanations) >= 20,                        f"{len(explanations)} explanations generated"),
    (all("predicted_risk" in e for e in explanations if e), "predicted_risk key present"),
    (all("top_features"   in e for e in explanations if e), "top_features key present"),
    (all("explanation"    in e for e in explanations if e), "explanation key present"),
    (all(isinstance(e.get("predicted_risk"), float) for e in explanations if e),
                                                     "predicted_risk is float"),
    (grounded_pct >= 0.80,                           f">= 80% grounded (got {grounded_pct:.0%})"),
], exercise_id="05b-2")


In [ ]:
evaluate(explain_prediction,
         "SHAP + LLM: predict cardiac risk, identify top SHAP features, explain in plain English.",
         exercise_id="05b-2")
progress()
